# Set 2 오답노트

## 검토한 파일

- `set_01_06_answer/02_question.ipynb`
- `set_01_06_answer/02_answer.ipynb`
- `set_01_06_answer/set_02_cbi.ipynb`
- `00_trying/01/02_question.ipynb`
- `00_trying/02/02_question.ipynb`

## 최종 답

- Q01: **3.4배**
- Q02: **0.999**
- Q03: **0.55**


## Q01 — 부가서비스 개수

### 틀리기 쉬운 부분

- `str.contains('Yes|No')`는 부분 문자열 검색이므로 `No internet service`도 `No`에 걸린다.
- 특정한 `No internet service`만 제거하기보다 문제 조건대로 **Yes/No가 아닌 모든 범주**를 제거해야 한다.
- 문제는 소수점 첫째 자리까지 요구하므로 `round(..., 1)`을 사용한다.
- `DataFrame.apply()`의 기본 `axis=0`은 함수를 열 단위로 적용한다. `apply()`가 항상 차원을 낮추는 것은 아니다.

### 권장 풀이

```python
df_q1 = df.copy()
cols_q1 = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

valid = df_q1[cols_q1].isin(['Yes', 'No']).all(axis=1)
df_q1 = df_q1.loc[valid].copy()
df_q1[cols_q1] = df_q1[cols_q1].replace({'Yes': 1, 'No': 0}).astype(int)
df_q1['service_count'] = df_q1[cols_q1].sum(axis=1)
service_freq = df_q1['service_count'].value_counts()
answer_q1 = round(service_freq.loc[1] / service_freq.loc[6], 1)
display(answer_q1)  # 3.4
```

`isin()`은 셀 전체 값을 정확히 비교한다. `.all(axis=1)`은 한 행의 6개 값이 모두 유효한지 검사한다. `replace()` 결과는 다시 대입해야 한다. `value_counts()`의 인덱스는 서비스 개수이므로 `.loc[1]`, `.loc[6]`으로 접근한다.


## Q02 — 피어슨 상관분석

### 틀리기 쉬운 부분

- 문제의 '나눈 값의 몫'은 `/`가 아니라 `//`이다. `/`의 결과도 이 데이터에서는 반올림 후 우연히 같은 답이 나오지만 조건을 지킨 풀이가 아니다.
- `corr()['tenure']`만 선택하면 tenure와 다른 변수의 관계만 확인한다. 문제는 세 변수의 모든 쌍 중 최댓값을 요구하므로 전체 상관행렬을 비교하는 편이 안전하다.
- 자기 자신과의 상관계수 1을 제외해야 한다.

### 권장 풀이

```python
df_q2 = df[['tenure', 'MonthlyCharges']].copy()
df_q2['used_month'] = df['TotalCharges'] // df['MonthlyCharges']

corr_abs = df_q2.corr(method='pearson').abs()
np.fill_diagonal(corr_abs.values, 0)
answer_q2 = round(corr_abs.max().max(), 3)
display(answer_q2)  # 0.999
```

`np.fill_diagonal()`에는 DataFrame의 값 배열인 `.values`를 전달한다. 숫자 하나에는 `abs(x)`, Series와 DataFrame에는 `.abs()`를 사용할 수 있다.

### 실제 데이터로 계산 과정 확인

일부 실제 데이터는 다음과 같다.

| index | tenure | MonthlyCharges | used_month |
|---:|---:|---:|---:|
| 0 | 1 | 29.85 | 1 |
| 1 | 34 | 56.95 | 33 |
| 2 | 2 | 53.85 | 2 |
| 3 | 45 | 42.30 | 43 |
| 7028 | 72 | 103.20 | 71 |
| 7031 | 66 | 105.65 | 64 |

실제 상관행렬:

|  | tenure | MonthlyCharges | used_month |
|---|---:|---:|---:|
| tenure | 1.000000 | 0.246862 | 0.998831 |
| MonthlyCharges | 0.246862 | 1.000000 | 0.246164 |
| used_month | 0.998831 | 0.246164 | 1.000000 |

상관행렬은 대칭이므로 같은 변수 쌍이 두 번 나타난다. 대각선의 1은 자기상관이므로 0으로 바꾸면 다음과 같다.

|  | tenure | MonthlyCharges | used_month |
|---|---:|---:|---:|
| tenure | 0.000000 | 0.246862 | 0.998831 |
| MonthlyCharges | 0.246862 | 0.000000 | 0.246164 |
| used_month | 0.998831 | 0.246164 | 0.000000 |

첫 번째 `.max()`의 실제 결과:

```python
corr_abs.max()

# tenure            0.998831
# MonthlyCharges    0.246862
# used_month        0.998831
```

두 번째 `.max()`는 위 열별 최댓값 중 전체 최댓값을 선택한다.

```python
corr_abs.max().max()
# 0.998830875...

round(corr_abs.max().max(), 3)
# 0.999
```

`corr()`에서 모든 쌍을 계산했더라도 `corr()['tenure']`를 선택하면 tenure가 포함된 관계만 남고 `MonthlyCharges ↔ used_month`는 버려진다. 모든 쌍의 최댓값을 구하려면 전체 행렬에서 대각선을 제거한 뒤 `.max().max()`를 사용한다. 이 데이터의 최댓값은 `tenure ↔ used_month`의 약 `0.998831`이다.


## Q03 — Logistic Regression과 F1-score

### 틀리기 쉬운 부분

- 첫 풀이에서는 문제의 `StreamingMovies` 대신 `StreamingTV`를 사용했다.
- Yes/No 외의 **모든 범주**를 -1로 바꿔야 하므로 특정 문자열만 나열하기보다 일반 변환 함수를 쓰는 편이 안전하다.
- 변환 함수는 숫자 열을 포함한 모든 분기에서 반드시 값을 `return`해야 한다.
- 변환된 DataFrame을 만들어 놓고 원본 DataFrame을 X와 y로 선택하면 변환이 모델에 반영되지 않는다.
- `df[cols_X]`를 수정하기 전 `.copy()`를 사용하면 `SettingWithCopyWarning`을 예방할 수 있다.
- 단일 y는 `df[['Churn']]`가 아니라 `df['Churn']`인 Series로 만든다.
- scaler는 학습 데이터에만 적합하고 평가 데이터에는 `transform()`만 적용한다. y는 정규화하지 않는다.

### 권장 풀이

```python
cols_X = [
    'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'MonthlyCharges', 'TotalCharges', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingMovies', 'PaperlessBilling'
]

def encode_binary(ser):
    if ser.dtype == 'object':
        return ser.map(lambda x: 1 if x == 'Yes' else 0 if x == 'No' else -1)
    return ser

X = df[cols_X].copy().apply(encode_binary)
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
answer_q3 = round(f1_score(y_test, y_pred), 2)
display(answer_q3)  # 0.55
```

seed 123은 데이터 분할의 `random_state`에 적용한다. 문제에서 모든 변수를 정규화하라는 말은 모든 독립변수를 의미하며 종속변수 y까지 정규화하라는 뜻이 아니다.


## 핵심 암기

1. 정확한 값 비교는 `isin()`, 부분 문자열 검색은 `str.contains()`를 사용한다.
2. 행의 모든 조건은 `.all(axis=1)`, 하나 이상의 조건은 `.any(axis=1)`이다.
3. 몫은 `//`, 일반 나눗셈은 `/`이다.
4. 상관행렬의 최댓값을 구할 때 대각선의 자기상관 1을 제외한다.
5. 문제에서 지정한 변수 이름과 개수를 모델링 전에 다시 확인한다.
6. 범주 변환 결과를 실제 X와 y에 사용하고 단일 y는 Series로 전달한다.
7. 정규화는 학습 데이터로 `fit`, 평가 데이터에는 `transform`만 한다.
